# Experiment Infrastructure

### Experiment Utility

In [1]:
from util.EventLogBuilder import EventLogBuilder
from util.experiment import *

# Domain Definition

In [2]:

# Activities
ACT_REGISTRATION = "ER Registration"
ACT_ER_TRIAGE = "ER Triage"
ACT_SEPSIS_TRIAGE = "Sepsis Triage"
ACT_TAKE_BLOOD = "Take Blood"
ACT_MEASURE_LEUK = "Measure Leukocytes"
ACT_MEASURE_LACTATE = "Measure Blood Lactate"
ACT_MEASURE_CRP = "Measure CRP"
ACT_ANTIMICROBIAL = "Administer Antimicrobials"
ACTIVITIES = [ACT_REGISTRATION, ACT_ER_TRIAGE, ACT_SEPSIS_TRIAGE, ACT_TAKE_BLOOD, ACT_MEASURE_LEUK, ACT_MEASURE_LACTATE, ACT_MEASURE_CRP, ACT_ANTIMICROBIAL]

# Case level PVs
SUSPECTED = "suspected"
DIAGNOSED = "diagnosed"
CAUSATIVE_PATHOGEN = "causative pathogen"
MDR_RISK = "multi-drug resistant risk"
# Task level PVs
AGE = "age"
NAME = "name"
INFECTION_SUSPECTED = "infection suspected"
HEART_RATE = "heart rate" # (bpm) ;  SIRS: > 90
BODY_TEMP = "body temperature" # (°C) ; SIRS: >38 | <36
RESP_RATE = "respiratory rate" # (b/min); SIRS: >20
LEUK_COUNT = "leukocyte count" # (M/l) SIRS >12 | <4
LACTATE = "blood lactate" # (mmol/l) 
NUM_SAMPLES = "no. blood samples"
ADMINISTERED_AGENT = "administered agent"

# Antimicrobials
# Antibiotica
VANCOMYCIN = "http://purl.bioontology.org/ontology/SNOMEDCT/372735009" # :is_antiMRSA true .  # No gram-negative coverage
CEFTRIAXONE = "http://purl.bioontology.org/ontology/SNOMEDCT/372670001" # :is_antiMRSA false ; :is_gram_negative true .
PIPERACILLIN = "http://purl.bioontology.org/ontology/SNOMEDCT/372836004" # :is_antiMRSA false ; :is_gram_negative true .
CEFEPIME = "http://purl.bioontology.org/ontology/SNOMEDCT/96048006" # :is_antiMRSA false ; :is_gram_negative true .
METRODINAZOLE = "http://purl.bioontology.org/ontology/SNOMEDCT/372602008" # both not
CLINDAMYCIN = "http://purl.bioontology.org/ontology/SNOMEDCT/372786004" # :is_antiMRSA true ; # No gram-negative coverage
ANTIBIOTICA = {VANCOMYCIN, CEFTRIAXONE, PIPERACILLIN, CEFEPIME, METRODINAZOLE}
MRSA_COVERAGE = {VANCOMYCIN, CLINDAMYCIN}
GRAM_NEGATIVE_COVERAGE = {CEFTRIAXONE, PIPERACILLIN, CEFEPIME}
# Antifungals
FLUCONAZOLE = "http://purl.bioontology.org/ontology/SNOMEDCT/387174006"
ECHINOCANDIN = "http://purl.bioontology.org/ontology/SNOMEDCT/373570003"
ANTIFUNGALS = {FLUCONAZOLE, ECHINOCANDIN}
ANTIMICROBIALS = ANTIBIOTICA | ANTIFUNGALS # A simplifcation; there are also, e.g., Antivirals, Antiparasitics, etc. 

# Diseases
SEPSIS = "sepsis"
SEPTIC_SHOCK = "septic shock"
DIABETES = "diabetes"

# MICROBIALS
MRSA = "http://purl.bioontology.org/ontology/SNOMEDCT/115329001" 
ECOLI = "http://purl.bioontology.org/ontology/SNOMEDCT/112283007" # A gram-negative bacterium that is common cause for sepsis
CANDIDA = "http://purl.bioontology.org/ontology/SNOMEDCT/53326005" # A fungus that is a common cause for sepsis
MICROBIALS = {MRSA, ECOLI, CANDIDA}
FUNGI = {CANDIDA}
NON_FUNGI = MICROBIALS - FUNGI

# Guideline Experiments

##### Constructor for case baseline pre diagnose, with SIRS criteria customizable

In [3]:
def base_case_pre_diagnose(infection, heart_rate, body_temp, resp_rate, leuk_count):
    log = (
        EventLogBuilder()
        .add_trace()
        .add_task(ACT_REGISTRATION).start("2024-06-02T07:55:00").complete("2024-06-02T07:59:00", {NAME : "John Maynard", AGE : 68})
        .add_task(ACT_ER_TRIAGE).start("2024-06-02T08:00:00").complete("2024-06-02T08:03:00", {
            INFECTION_SUSPECTED : infection, 
        })
        .add_task(ACT_SEPSIS_TRIAGE).start("2024-06-02T08:03:00").complete("2024-06-02T08:10:00", {
            HEART_RATE : 110 if heart_rate else 78, 
            BODY_TEMP : 39 if body_temp else 37, 
            RESP_RATE : 28 if resp_rate else 19, 
        })
        .add_task(ACT_TAKE_BLOOD).start("2024-06-02T08:02:00").complete("2024-06-02T08:03:00", {NUM_SAMPLES : 3})
        .add_task(ACT_MEASURE_LEUK).start("2024-06-02T08:04:00").complete("2024-06-02T08:08:00", {LEUK_COUNT : 34 if leuk_count else 8})
    )
    return log


log_xes = base_case_pre_diagnose(True, True, False, False, True)
print(log_xes.to_xes())

<?xml version="1.0" encoding="UTF-8"?>
<log xes.version="2.0" xmlns="http://www.xes-standard.org/">
	<string key="concept:name" value="example_log" />
	<trace>
		<string key="concept:name" value="trace_1" />
		<event>
			<string key="concept:name" value="ER Registration" />
			<string key="concept:instance" value="ER Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T09:55:00+02:00" />
			<string key="lifecycle:transition" value="start" />
		</event>
		<event>
			<string key="concept:name" value="ER Registration" />
			<string key="concept:instance" value="ER Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T09:59:00+02:00" />
			<string key="lifecycle:transition" value="complete" />
			<string key="name" value="John Maynard" />
			<int key="age" value="68" />
		</event>
		<event>
			<string key="concept:name" value="ER Triage" />
			<string key="concept:instance" value="ER Triage_1_2" />
			<date key="time:timestamp" value="2024-06-02T10:00:00+02:0

##### Constructor for case baseline of patient after triage

In [4]:
def triaged_patient_base(case_attr=dict()): 
    log = (
        base_case_pre_diagnose(True, True, False, True, True)
        .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:11:00").complete("2024-06-02T08:16:00", {LACTATE : 2.4})
    )
    log._current_trace["case_attributes"] = case_attr
    return log


log_xes = triaged_patient_base({DIAGNOSED : [DIABETES, SEPSIS, SEPTIC_SHOCK]})
print(log_xes.to_xes())

<?xml version="1.0" encoding="UTF-8"?>
<log xes.version="2.0" xmlns="http://www.xes-standard.org/">
	<string key="concept:name" value="example_log" />
	<trace>
		<string key="concept:name" value="trace_1" />
		<list key="diagnosed">
			<values>
				<string key="diagnosed" value="diabetes" />
				<string key="diagnosed" value="sepsis" />
				<string key="diagnosed" value="septic shock" />
			</values>
		</list>
		<event>
			<string key="concept:name" value="ER Registration" />
			<string key="concept:instance" value="ER Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T09:55:00+02:00" />
			<string key="lifecycle:transition" value="start" />
		</event>
		<event>
			<string key="concept:name" value="ER Registration" />
			<string key="concept:instance" value="ER Registration_1_0" />
			<date key="time:timestamp" value="2024-06-02T09:59:00+02:00" />
			<string key="lifecycle:transition" value="complete" />
			<string key="name" value="John Maynard" />
			<int key="age" 

## Blood Lactate

### "For adults suspected of having sepsis, we suggest measuring blood lactate." ...
... Suspicion of sepsis is governed by a suspected infection and the presence of at least two of the SIRS criteria (i) body temperature outside of [36°C;38°C], (ii) heartrate > 90bpm, (iii) respiratory rate > 20 breaths/minute, and (iv) leukocyte count outside of [4M/l;12M/l]

In [5]:
def measure_lactate_base_experiment(system_under_eval):
    experiment = Experiment(system_under_eval)
    experiment.test_two_classes([
        base_case_pre_diagnose(False, False, False, False, False), # baseline
        base_case_pre_diagnose(False, False, True, True, False), # no infection
        base_case_pre_diagnose(True, True, False, False, False), # not sufficient SIRS
        base_case_pre_diagnose(True, True, True, True, True) # has already been measured
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:09:00").complete("2024-06-02T08:12:00", {LACTATE : 1.6})
    ], [
        base_case_pre_diagnose(True, True, True, True, True), # baseline
        base_case_pre_diagnose(True, False, True, True, False), # body temp and resp
        base_case_pre_diagnose(True, True, False, False, True) # heart rate and leuk
    ], ACT_MEASURE_LACTATE)
    return experiment.results()

### "For adults with sepsis or septic shock, we suggest guiding resuscitation to decrease serum lactate in patients with elevated lactate levels over not using serum lactate." ...
...  "Measure the lactate of patients [...] again [every] 2-4 hours if the [last] result is greater than 2mmol/L"

In [16]:
def measure_lactate_time_experiment(system_under_eval):
    experiment = Experiment(system_under_eval)

    # Check that it's repeated if two hours have passed and the last measurement was > 2 mmol/L
    experiment.test_two_classes([
        (base_case_pre_diagnose(True, True, True, False, False) # Has been neither long enough ago nor high enough
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 1.6})
            .set_current_time("2024-06-02T09:15:00")), 
        (base_case_pre_diagnose(True, True, True, False, False) # Has been long enough ago but not high enough
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 1.6})
            .set_current_time("2024-06-02T10:20:00")), 
        (base_case_pre_diagnose(True, True, True, False, False) # Has been high but not long enough ago
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .set_current_time("2024-06-02T09:15:00")), 
        (base_case_pre_diagnose(True, True, True, False, False) # Has been long enough and high enough in any, but not high enough in the last measurement
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T11:10:00").complete("2024-06-02T11:15:00", {LACTATE : 1.8})
            .set_current_time("2024-06-02T14:00:00")),
        (base_case_pre_diagnose(True, True, True, False, False) # Has been long enough and high enough in any, but not long enough the last measurement
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T11:10:00").complete("2024-06-02T11:15:00", {LACTATE : 2.6})
            .set_current_time("2024-06-02T12:15:00"))
    ], [
        (base_case_pre_diagnose(True, True, True, False, False) # Has been high a while ago
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .set_current_time("2024-06-02T10:20:00")), 
        (base_case_pre_diagnose(True, True, True, False, False) # Has been continuously high
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T08:10:00").complete("2024-06-02T08:15:00", {LACTATE : 2.5})
            .add_task(ACT_MEASURE_LACTATE).start("2024-06-02T11:10:00").complete("2024-06-02T11:15:00", {LACTATE : 2.2})
            .set_current_time("2024-06-02T14:00:00"))
    ], ACT_MEASURE_LACTATE)

    
    # Extra Test: Check that it's more important than a non-urgent measurement
    for result in experiment.high_results:
        experiment.test_optional(result.get(ACT_MEASURE_LACTATE, 0) > result.get(ACT_MEASURE_CRP, 0)) 

    return experiment.results()


## Antimicrobials

### "For adults with sepsis or septic shock at high risk of [methicillin-resistant Staphylococcus aureus (MRSA)], ... 
... we recommend using empiric antimicrobials with MRSA coverage over using antimicrobials without MRSA coverage [and] at low risk of MRSA, we suggest against using empiric antimicrobials with MRSA coverage, as compared with using antimicrobials without MRSA coverage."

In [7]:
def antimicrobials_MRSA_experiment(system_under_eval):
    experiment = Experiment(system_under_eval)
    experiment.test_two_classes([
        triaged_patient_base({DIAGNOSED : [SEPSIS]}),
        triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : ECOLI})
    ], [
        triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : MRSA})
    ], *map(lambda x: (ACT_ANTIMICROBIAL, x), MRSA_COVERAGE))

    # Additional validation outside of A-B schema
    f_A = experiment.low_results[0]
    f_B = experiment.high_results[0]
    for mrsa_cover in MRSA_COVERAGE:
        experiment.test_optional((0, '') > f_A.get((ACT_ANTIMICROBIAL, mrsa_cover), (0, ''))) # TODO Fixed number
        for other in ANTIMICROBIALS - MRSA_COVERAGE:
            experiment.test_optional(f_B.get((ACT_ANTIMICROBIAL, mrsa_cover), (0, '')) > f_B.get((ACT_ANTIMICROBIAL, other), (0, '')))

    return experiment.results()

### "For adults with sepsis or septic shock and high risk for multidrug-resistant (MDR) organisms, ...
... we suggest using 2 antimicrobials with gram-negative coverage for empiric treatment over 1 gram-negative agent [and, for] low risk for multidrug-resistant (MDR) organisms, we suggest against using 2 gram-negative agents for empiric treatment, as compared to 1 gram-negative agent. [...] We suggest against using double gram-negative coverage once the causative pathogen and the susceptibilities are known." 

In [8]:
def antimicrobials_MDR_experiment(system_under_eval): #TODO: what if already administered two, but the one under eval is one of those?
    experiment = Experiment(system_under_eval)
    experiment.test_two_classes([
        triaged_patient_base({DIAGNOSED : [SEPSIS]}), # Baseline for no double treatment, no increased risk
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True}) # Already administered two gram-negative antibiotics
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:05:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T08:30:00")
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:15:00", {ADMINISTERED_AGENT : PIPERACILLIN}).complete("2024-06-02T09:30:00"),
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True, CAUSATIVE_PATHOGEN : ECOLI}) # Only administered one gram-negative antibiotic, but causative pathogen is known
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:05:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T08:30:00")
    ], [
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True}), # Baseline for double treatment, increased risk, no antibiotics administered so far
        triaged_patient_base({DIAGNOSED : [SEPSIS], MDR_RISK : True}) # Only one gram-negative antibiotic administered so far (although twice)
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T08:05:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T08:30:00")
            .add_task(ACT_ANTIMICROBIAL).start("2024-06-02T09:00:00", {ADMINISTERED_AGENT : CEFTRIAXONE}).complete("2024-06-02T09:30:00")
    ], *[(ACT_ANTIMICROBIAL, x) for x in GRAM_NEGATIVE_COVERAGE])
    return experiment.results()

### "For adults with sepsis or septic shock at high risk of fungal infection, ... 
... we suggest using empiric antifungal therapy over no antifungal therapy [and] at low risk of fungal infection, we suggest against empiric use of antifungal therapy."

In [9]:
def antimicrobials_Fungi_experiment(system_under_eval):
    experiment = Experiment(system_under_eval)

    experiment.test_two_classes([
        *[triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : x}) for x in NON_FUNGI],
        triaged_patient_base({DIAGNOSED : [SEPSIS]}) # Pathogen unknown
    ], [
        *[triaged_patient_base({DIAGNOSED : [SEPSIS], CAUSATIVE_PATHOGEN : x}) for x in FUNGI] 
    ], *[(ACT_ANTIMICROBIAL, x) for x in ANTIFUNGALS])
    
    # Additional validation outside of A-B schema
    f_A = experiment.low_results[0]
    f_B = experiment.high_results[0]
    for antifungal in ANTIFUNGALS:
        experiment.test_optional(f_A.get((ACT_ANTIMICROBIAL, antifungal), (0, '')) < (0, '')) # TODO Fixed number
        for other in ANTIMICROBIALS - ANTIFUNGALS:
            experiment.test_optional(f_B.get((ACT_ANTIMICROBIAL, antifungal), (0, '')) > f_B.get((ACT_ANTIMICROBIAL, other), (0, '')))

    return experiment.results()

# Karibdis

#### Experiment Bridge

In [10]:
import os
import sys
karibdis_path = os.path.abspath('../../src')
print(karibdis_path)
sys.path.insert(0, karibdis_path)

c:\unsynchronized\projects\.Graph_Workbench\Workbench\karibdis_project\karibdis\src


In [11]:
%load_ext autoreload
%autoreload 2
from karibdis.KnowledgeGraphBPMS import KnowledgeGraphBPMS
from karibdis.KnowledgeImporter import OnlineEventImporter
from rdflib import Graph, RDF
from karibdis.utils import BASE_PROCESS_ONTOLOGY as BPO
from karibdis import get_logger as karibdis_logger

In [12]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from rdflib import Graph, Variable

In [23]:
class Karibdis_for_Evaluation(SystemUnderEvaluation):

    def __init__(self, activities):
        karibdis_logger().setLevel('WARNING')
        self.bpms = KnowledgeGraphBPMS() 
        self.bpms.pkg += Graph().parse('additional_knowledge.ttl', format='turtle')
        self.bpms.pkg += Graph().parse('snomed_antimicrobials.ttl', format='turtle')
        self.bpms.engine.deduce_chained()

        pre_loader = OnlineEventImporter(self.bpms.pkg)
        self.activity_map = {}
        for activity in activities:
            activity_uri = pre_loader.activity_node(activity)
            self.bpms.pkg.add((activity_uri, RDF.type, BPO.Activity))
            self.activity_map[activity_uri] = activity
        # print(f'Mapping Activity URIs: {activity_map}')

    def load_log_prefix(self, log_xes):
        x = xes_importer.deserialize(log_xes, {"show_progress_bar": False})
        loader = OnlineEventImporter(self.bpms.pkg)
        for trace in x:
            # print(trace)
            loader.case_attributes = trace._attributes.keys() - {'concept:name'}
            for case_attr in loader.case_attributes:
                trace[0][case_attr] = trace._attributes[case_attr]
            for event in trace:
                event['case:concept:name'] = trace._attributes['concept:name']
                for key, value in event.items():
                    if hasattr(value, "items") and value['children']:
                        event[key] = list(zip(*value['children']))[1]
                # print(event)
                loader.translate_event(event) # TODO: Potentially use existing load-whole-log-function
        loader.load()

    def get_valuations(self, log_xes, current_time):
        triples_before = list(self.bpms.pkg)
        try:
            with mock_time(current_time):
                self.load_log_prefix(log_xes)
                self.bpms.engine.deduce_chained()
                decision = next(self.bpms.engine.open_decisions())
                valuation, options, reasoning = list(zip(*decision.get_top_k_results())) # TODO translate back from URIs
        finally:
            self.bpms.pkg -= self.bpms.pkg
            self.bpms.pkg += triples_before
        # print(valuation, options)
        def translate_option(option):
            option_key, effect = option
            result = self.activity_map.get(option_key['activity'])
            if len(option_key.keys()) > 1:
                result = tuple([result, *map(lambda key: option_key[key].toPython(), sorted(option_key.keys() - {Variable('activity')}))])
            return result 
        return dict(zip(map(translate_option, options), zip(valuation, reasoning)))


### Run Experiments

In [24]:
measure_lactate_base_experiment(Karibdis_for_Evaluation(ACTIVITIES))

Karibdis_for_Evaluation - measure_lactate_base_experiment
{'Sepsis Triage': (0, []), 'Take Blood': (0, []), 'Administer Antimicrobials': (0, []), 'ER Registration': (0, []), 'Measure CRP': (0, []), 'ER Triage': (0, []), 'Measure Blood Lactate': (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/373219008'): (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/372602008'): (0, []), 'Measure Leukocytes': (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/419241000'): (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/372836004'): (-2, ['Additional gram-negative antimicrobials should not be used, as  although only 0 gram-negative antimicrobials have been administered so far and the risk for multi-drug resistance is not highalthough the causative pathogen is unknown.']), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/3871740

('Karibdis_for_Evaluation - measure_lactate_base_experiment (4x3x1)', 12, 12)

In [25]:
measure_lactate_time_experiment(Karibdis_for_Evaluation(ACTIVITIES))

Karibdis_for_Evaluation - measure_lactate_time_experiment
{'ER Registration': (0, []), 'Measure Blood Lactate': (0, []), 'Measure Leukocytes': (0, []), 'Sepsis Triage': (0, []), 'Take Blood': (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/373219008'): (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/419241000'): (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/372602008'): (0, []), 'Administer Antimicrobials': (0, []), 'ER Triage': (0, []), 'Measure CRP': (0, []), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/372670001'): (-2, ['Additional gram-negative antimicrobials should not be used, as  although only 0 gram-negative antimicrobials have been administered so far and the risk for multi-drug resistance is not highalthough the causative pathogen is unknown.']), ('Administer Antimicrobials', 'http://purl.bioontology.org/ontology/SNOMEDCT/3728360

('Karibdis_for_Evaluation - measure_lactate_time_experiment (5x2x1)', 10, 10)

In [18]:
antimicrobials_MRSA_experiment(Karibdis_for_Evaluation(ACTIVITIES))

Karibdis_for_Evaluation - antimicrobials_MRSA_experiment
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -2, -2, -2, -2, -2, -3, -3) (({rdflib.term.Variable('anti'): rdflib.term.URIRef('http://purl.bioontology.org/ontology/SNOMEDCT/373219008'), rdflib.term.Variable('activity'): rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials')}, [(rdflib.term.URIRef('http://example.org/Task_trace_1_7'), rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/instanceOf'), rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials')), (rdflib.term.URIRef('http://example.org/Task_trace_1_7'), rdflib.term.URIRef('http://example.org/ProcessValue_administered%20agent'), rdflib.term.URIRef('http://purl.bioontology.org/ontology/SNOMEDCT/373219008'))]), ({rdflib.term.Variable('activity'): rdflib.term.URIRef('http://example.org/Activity_ER%20Triage')}, [(rdflib.term.URIRef('http://example.org/Task_trace_1_7'), rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontolo

('Karibdis_for_Evaluation - antimicrobials_MRSA_experiment (2x1x2)', 4, 4)

In [19]:
antimicrobials_MDR_experiment(Karibdis_for_Evaluation(ACTIVITIES))

Karibdis_for_Evaluation - antimicrobials_MDR_experiment
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -2, -2, -2, -2, -2, -3, -3) (({rdflib.term.Variable('activity'): rdflib.term.URIRef('http://example.org/Activity_ER%20Registration')}, [(rdflib.term.URIRef('http://example.org/Task_trace_1_7'), rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/instanceOf'), rdflib.term.URIRef('http://example.org/Activity_ER%20Registration'))]), ({rdflib.term.Variable('anti'): rdflib.term.URIRef('http://purl.bioontology.org/ontology/SNOMEDCT/372602008'), rdflib.term.Variable('activity'): rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials')}, [(rdflib.term.URIRef('http://example.org/Task_trace_1_7'), rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/baseontology/instanceOf'), rdflib.term.URIRef('http://example.org/Activity_Administer%20Antimicrobials')), (rdflib.term.URIRef('http://example.org/Task_trace_1_7'), rdflib.term.URIRef('http://example.org/ProcessValue_administered

('Karibdis_for_Evaluation - antimicrobials_MDR_experiment (3x2x3)', 18, 18)